# Week 1 Lab — Meet QuickCart's raw data

**Your role:** it is your first day as a Junior Data Engineer at QuickCart. The mobile-app team has sent the data team three days of exports. Before anyone builds a pipeline, you must **look at the raw data like an engineer** and write down every problem that could break a pipeline or a report.

**How to use this notebook**
- Run the cells **from top to bottom** with **Shift + Enter**.
- ✏️ **TODO** cells: replace every `___` with the right code, then run the cell.
- 🧠 **PREDICT** questions: type your prediction *before* you run the next cell.
- Type your answers in the *Your answer:* lines (double-click a text cell to edit it).
- If you see `NameError: name '___' is not defined`, you have not filled in a blank yet.

**At the end** you will copy your five most important problems into `docs/data_issues.md`.

## Part 0 — Find the data files
The generator you ran in Step 3 wrote the files into `data/source_exports/`. This notebook lives in the `notebooks/` folder, so the path goes **up one level** (`..`) and then into `data/source_exports`.

In [ ]:
# Part 0 - tell Python where the data files are
import json                   # reads JSON files
from pathlib import Path      # builds file paths that work on Windows AND macOS
import pandas as pd           # tables (DataFrames)

DATA_DIR = Path("..") / "data" / "source_exports"
print("Looking in:", DATA_DIR.resolve())

for file in sorted(DATA_DIR.glob("*")):
    if not file.name.startswith("."):          # skip hidden files such as .gitkeep
        print(f"{file.name:28} {file.stat().st_size / 1024:8.0f} KB")

## Part A — Look at a file as raw text
Before using any library, look at what the source system *actually* sent. This is what an engineer does first.

In [ ]:
first_file = DATA_DIR / "orders_2026-10-01.json"
raw_text = first_file.read_text(encoding="utf-8")   # the whole file as one long piece of text

print("Characters in the file:", len(raw_text))
print(raw_text[:900])                                 # only the first 900 characters

**Question A1.** Is this data *structured*, *semi-structured* or *unstructured*? Give one reason.

*Your answer:* 

## Part B — Open the JSON with Python
`json.load` turns JSON text into ordinary Python objects: `{ }` becomes a **dict** and `[ ]` becomes a **list**.

In [ ]:
with open(first_file, encoding="utf-8") as f:
    export = json.load(f)                 # JSON text -> Python dict

print(type(export))
print(list(export.keys()))

In [ ]:
# ✏️ TODO B1: print the metadata of the export. Replace ___ with the right key name.
print("export_date  :", export["export_date"])
print("source_system:", export["source_system"])
print("record_count :", export[___])

orders_list = export["orders"]            # the list of orders
print("orders in the list:", len(orders_list))

In [ ]:
# Look at ONE order, printed with indentation so it is easy to read
print(json.dumps(orders_list[0], indent=2))

**Question B2.** In the order above, name one field that is a *simple value*, one that is a *nested object* `{ }` and one that is a *list* `[ ]`.

*Your answer:* 

**Question B3.** One order can contain many items. What kind of relationship is that, and how would you store it in a relational database?

*Your answer:* 

## Part C — Put the orders into a DataFrame

In [ ]:
orders = pd.DataFrame(orders_list)
print("rows, columns:", orders.shape)
orders.head(3)

**Question C1.** Look at the columns `payment`, `delivery` and `items`. Why is this not yet a clean table that you could load into a database or give to an analyst?

*Your answer:* 

## Part D — Hunt for data problems (the main task)
Each check below looks for one kind of problem. Write down the numbers you get — they are different for every student because everyone uses their own seed.

### D1 — Is `order_id` really unique?
🧠 **PREDICT:** an order ID should identify exactly one order. How many duplicate rows do you expect?

*My prediction:* 

In [ ]:
number_of_rows = len(orders)
number_of_unique_ids = orders["order_id"].nunique()     # nunique = number of DIFFERENT values

print("rows:", number_of_rows)
print("unique order_ids:", number_of_unique_ids)
print("duplicate rows:", number_of_rows - number_of_unique_ids)

In [ ]:
# Show some duplicated orders next to each other
duplicated_rows = orders[orders["order_id"].duplicated(keep=False)]    # keep=False marks ALL copies
duplicated_rows.sort_values(["order_id", "updated_at"])[["order_id", "status", "updated_at"]].head(6)

**Question D1.** Are the two copies of an order identical? Which copy should a revenue report use, and how can you tell?

*Your answer:* 

### D2 — Missing customers

In [ ]:
# ✏️ TODO D2: count the orders that have NO customer_id.
# Hint: .isna() gives True for every missing value, and .sum() counts the Trues.
missing_customers = orders["customer_id"].___
print("orders without customer_id:", missing_customers)

### D3 — Order status values
🧠 **PREDICT:** how many *different* status values do you expect in a well-designed system?

*My prediction:* 

In [ ]:
orders["status"].value_counts()

**Question D3.** How many status values are there really? What would happen to a report that counts delivered orders with `status == "delivered"`?

*Your answer:* 

### D4 — Items and quantities
Items are *inside* each order. To check them we build one row per item — the same one-to-many idea as Question B3.

In [ ]:
item_rows = []                                   # an empty list that we will fill
for order in orders_list:                        # go through every order...
    for item in order["items"]:                  # ...and every item inside that order
        item_rows.append({
            "order_id": order["order_id"],       # remember which order the item belongs to
            "product_id": item["product_id"],
            "quantity": item["quantity"],
            "unit_price": item["unit_price"],
        })

items = pd.DataFrame(item_rows)
print("item rows:", len(items))
items.describe()

In [ ]:
# ✏️ TODO D4: keep only the items whose quantity is 0 or less.
# Replace ___ with a comparison operator.
bad_quantities = items[items["quantity"] ___ 0]
print("items with quantity <= 0:", len(bad_quantities))
bad_quantities.head()

In [ ]:
# Orders that contain no items at all
empty_orders = orders[orders["items"].apply(len) == 0]     # len() of each order's list of items
print("orders with no items:", len(empty_orders))

### D5 — Does the order total match its items?
`payment.total_amount` should equal the sum of *quantity × unit_price* of the items.

In [ ]:
mismatches = 0
for order in orders_list:
    items_total = sum(item["quantity"] * item["unit_price"] for item in order["items"])
    paid_total = order["payment"]["total_amount"]
    if abs(items_total - paid_total) > 0.01:        # allow for rounding of 1 fils
        mismatches += 1

print("orders where total_amount does not match the items:", mismatches)

### D6 — The customer file

In [ ]:
# Phone numbers are identifiers, not numbers: read them as text so nothing is lost
customers = pd.read_csv(DATA_DIR / "customers.csv", dtype={"phone": str})
print("rows, columns:", customers.shape)
customers.head()

In [ ]:
customers.isna().sum()          # missing values per column

In [ ]:
customers["zone"].value_counts(dropna=False)     # dropna=False also shows missing values (NaN)

In [ ]:
# ✏️ TODO D6: how many customers have the same e-mail as another customer
# if we ignore capital letters? Replace ___ with the pandas method that marks repeated values.
same_email = customers["email"].str.lower().___().sum()
print("customers registered twice (same e-mail):", same_email)

In [ ]:
customers["phone"].head(12)      # look carefully at the formats

**Question D6.** List the problems you can see in the customer file. Which of its columns contain **personal data** that must be protected?

*Your answer:* 

### D7 — The product file

In [ ]:
products = pd.read_csv(DATA_DIR / "products.csv")
print(products.dtypes)
products["category"].value_counts()

In [ ]:
# Which prices cannot be turned into a number?
# errors="coerce" turns anything that is not a valid number into NaN (missing)
price_as_number = pd.to_numeric(products["unit_price"], errors="coerce")
products[price_as_number.isna()]

**Question D7.** Why is the `unit_price` column not stored as a number, and why is that dangerous for a revenue calculation?

*Your answer:* 

## Part E — Your turn (no template)
Write code that goes through **all** `orders_*.json` files and prints, for each file:
1. the file name,
2. the number of order records,
3. the number of **unique** order IDs,
4. the number of orders with a missing `customer_id`.

*Hint:* `sorted(DATA_DIR.glob("orders_*.json"))` gives you the files; reuse ideas from Parts B and D.

In [ ]:
# Part E - write your code here


### ⭐ Stretch (optional)
Real systems sometimes send data **late**. Check whether any order in a file happened on a *different day* from the file's `export_date`.
To see an example, first generate day 11 in the terminal: `python tools/generate_day.py --seed <your seed> --date 2026-10-11`

In [ ]:
# Stretch - write your code here


## Part F — Record your findings
Open `docs/data_issues.md` and fill in **at least five** problems, using **your own** numbers from this notebook. Then answer the reflection question at the bottom of that file.

Finally, save this notebook (**Ctrl + S** on Windows, **Cmd + S** on macOS) and go back to the lab guide for the Git step.